# Taller integrador — Validación y Administración de Requerimientos
**Proyecto:** SINO · Sistema de Causación de Ingresos No Operacionales
**Asignatura:** Gestión de Requisitos de Software
**Repositorio:** https://github.com/Manum03/Taller_requi

Este notebook cubre:
- **Parte 6.** Matriz de trazabilidad en Python y detección de trazabilidad incompleta.
- **Parte 15.** Control de cambios: métricas automáticas y gráficos.

Los datos salen de los 8 requerimientos seleccionados (RF-02, RF-05, RF-07, RF-09, RF-11, RNF-03, RNF-04, RNF-09) y del historial git real del proyecto.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_colwidth', 80)

---
## Parte 6. Matriz de trazabilidad en Python
Necesidad → Requerimiento → Historia de usuario → Caso de uso → Diseño → Código → Prueba

La columna **Código** apunta al módulo y a las funciones de `index.html` que implementan cada requerimiento. Una celda vacía (`""`) significa que ese artefacto todavía no existe.

In [ ]:
datos = [
  {
    "Requisito": "RF-02",
    "Tipo": "Funcional",
    "Descripcion": "Registro de terceros (bancos/clientes) con NIT, DV calculado y estado.",
    "Historia": "US-02",
    "Caso_Uso": "CU-02",
    "Diseño": "DS-02",
    "Codigo": "index.html › Maestros.guardarTercero(), validarCampoTercero(), Util.calcDV()",
    "Prueba": "TC-01"
  },
  {
    "Requisito": "RF-05",
    "Tipo": "Funcional",
    "Descripcion": "Registro de causación manual con autocompletado de cuentas, periodo Desde/Hasta y cálculo de retención.",
    "Historia": "US-05",
    "Caso_Uso": "CU-05",
    "Diseño": "DS-05",
    "Codigo": "index.html › Causaciones.guardarManual(), calcularAsiento(), recalcular()",
    "Prueba": "TC-02"
  },
  {
    "Requisito": "RF-07",
    "Tipo": "Funcional",
    "Descripcion": "Flujo de aprobación con doble control (Maker-Checker).",
    "Historia": "US-07",
    "Caso_Uso": "CU-07",
    "Diseño": "DS-07",
    "Codigo": "index.html › Causaciones.aprobar(), rechazar(); Views.renderBandeja()",
    "Prueba": "TC-03"
  },
  {
    "Requisito": "RF-09",
    "Tipo": "Funcional",
    "Descripcion": "Libro auxiliar con filtros por tercero, tipo y fecha, exportable a CSV/Excel.",
    "Historia": "US-09",
    "Caso_Uso": "CU-09",
    "Diseño": "DS-09",
    "Codigo": "index.html › Reportes.renderLibroAuxiliar(), exportCSV()",
    "Prueba": ""
  },
  {
    "Requisito": "RF-11",
    "Tipo": "Funcional",
    "Descripcion": "Estado de cuenta por tercero con causado, cobrado y saldo pendiente.",
    "Historia": "US-11",
    "Caso_Uso": "",
    "Diseño": "DS-11",
    "Codigo": "index.html › Reportes.renderEstadoCuenta()",
    "Prueba": ""
  },
  {
    "Requisito": "RNF-03",
    "Tipo": "No funcional",
    "Descripcion": "RBAC estricto: cada vista y botón valida el rol activo antes de ejecutar.",
    "Historia": "",
    "Caso_Uso": "",
    "Diseño": "DS-03",
    "Codigo": "index.html › UI.applyRoleVisibility(); Maestros.puedeGestionarTerceros()",
    "Prueba": "TC-04"
  },
  {
    "Requisito": "RNF-04",
    "Tipo": "No funcional",
    "Descripcion": "Procesar y contabilizar 10.000 causaciones de cierre en menos de 5 minutos.",
    "Historia": "",
    "Caso_Uso": "",
    "Diseño": "DS-04",
    "Codigo": "index.html › Batch.generar(), contabilizarLote()",
    "Prueba": "TC-05"
  },
  {
    "Requisito": "RNF-09",
    "Tipo": "No funcional",
    "Descripcion": "Prevención de errores: 'Guardar' deshabilitado hasta que Débitos = Créditos.",
    "Historia": "US-05",
    "Caso_Uso": "CU-05",
    "Diseño": "DS-05",
    "Codigo": "index.html › Causaciones.calcularAsiento(), recalcular()",
    "Prueba": "TC-06"
  }
]

df = pd.DataFrame(datos)
display(df)

### Función que identifica trazabilidad incompleta
Revisa, para cada requerimiento, si le falta la historia, el caso de uso, el código o la prueba.

In [ ]:
ARTEFACTOS = {
    "Historia": "sin historia",
    "Caso_Uso": "sin caso de uso",
    "Codigo":   "sin código",
    "Prueba":   "sin prueba",
}

def vacio(valor):
    return pd.isna(valor) or str(valor).strip() == ""

def trazabilidad_incompleta(df):
    """Devuelve (resumen, detalle).
    resumen: dict {categoría: [IDs de requisitos]}
    detalle: DataFrame con los requisitos que tienen al menos un artefacto faltante."""
    resumen = {}
    for col, etiqueta in ARTEFACTOS.items():
        resumen[etiqueta] = df.loc[df[col].apply(vacio), "Requisito"].tolist()

    faltantes = df.apply(
        lambda fila: [ARTEFACTOS[c] for c in ARTEFACTOS if vacio(fila[c])], axis=1)
    detalle = df.assign(Faltantes=faltantes.apply(", ".join))
    detalle = detalle[faltantes.apply(len) > 0][["Requisito", "Tipo", "Faltantes"]]
    return resumen, detalle.reset_index(drop=True)

resumen, detalle = trazabilidad_incompleta(df)

for categoria, ids in resumen.items():
    print(f"Requerimientos {categoria:<16}: {', '.join(ids) if ids else 'ninguno'}")

print("\nElementos con trazabilidad incompleta:")
display(detalle)

### Estado de cobertura de la matriz

In [ ]:
df["Estado"] = df.apply(
    lambda f: "Cubierto" if all(not vacio(f[c]) for c in ARTEFACTOS) else "Incompleto", axis=1)
display(df[["Requisito", "Historia", "Caso_Uso", "Diseño", "Codigo", "Prueba", "Estado"]])

cobertura = {etq: round(100 * (1 - len(ids) / len(df)), 1) for etq, ids in resumen.items()}
print("% de requerimientos que SÍ tienen cada artefacto:")
for etq, pct in cobertura.items():
    print(f"  {etq.replace('sin ', 'con '):<16} {pct:>5}%")
print(f"\nCubiertos de punta a punta: {(df['Estado']=='Cubierto').sum()} de {len(df)}")

### Pregunta: ¿qué riesgo existe si un requerimiento está implementado pero no tiene prueba asociada?
En SINO, **RF-09 (Libro auxiliar)** y **RF-11 (Estado de cuenta)** tienen código pero no tienen prueba. El riesgo es que nadie puede **demostrar objetivamente** que funcionan.
- **Errores que llegan al usuario sin detectarse.** El libro auxiliar podría omitir causaciones contabilizadas o exportar mal los decimales en el CSV, y el estado de cuenta podría mostrar un saldo pendiente incorrecto. Son reportes que un contador usa para tomar decisiones o responder a un tercero.
- **Regresiones invisibles.** El CR-01 cambió las fechas Desde/Hasta y RF-09 filtra por fecha. Sin una prueba no hay forma de saber si ese cambio rompió el filtro del libro auxiliar.
- **El requisito no se puede dar por "Validado".** Según el plan de administración, un requerimiento pasa a *Validado* solo si tiene una prueba que lo verifica. Sin ella se queda en *Implementado*.
- **Sin evidencia para auditoría o cumplimiento.** RNF-07 (NIIF/DIAN) exige reportes confiables, y sin prueba no hay evidencia que mostrar.

**Acción:** crear TC-07 (RF-09) y TC-08 (RF-11) y documentar CU-11 antes del próximo ciclo.

---
## Parte 15. Control de cambios en Python
Lista de solicitudes de cambio (CR). CR-01 a CR-09 salen del historial git real: commits del **22 y 23 de octubre de 2026** ("cambios solicitados"). CR-SEG-01 es el cambio de seguridad de la Parte 16 y está en análisis.

In [ ]:
cambios = [
  {
    "id": "CR-01",
    "requisito": "RF-05",
    "fecha": "2026-10-22",
    "descripcion": "Agregar Fecha Vencimiento (Hasta) a cada causación y corregir el filtro Desde/Hasta (RN-CAU-01)",
    "version_anterior": "1.0",
    "version_nueva": "1.1",
    "impacto": "Alto",
    "prioridad": "Alta",
    "estado": "Aprobado"
  },
  {
    "id": "CR-02",
    "requisito": "RF-02",
    "fecha": "2026-10-22",
    "descripcion": "Validar RUT, NIT 8–10 dígitos, correo y teléfono; bloquear NIT duplicado (RN-TER-01)",
    "version_anterior": "1.0",
    "version_nueva": "1.1",
    "impacto": "Alto",
    "prioridad": "Alta",
    "estado": "Aprobado"
  },
  {
    "id": "CR-03",
    "requisito": "RF-02",
    "fecha": "2026-10-22",
    "descripcion": "Solo el Analista Contable crea/edita terceros; Coordinador y Admin en solo lectura (RN-TER-02)",
    "version_anterior": "1.1",
    "version_nueva": "1.2",
    "impacto": "Medio",
    "prioridad": "Alta",
    "estado": "Aprobado"
  },
  {
    "id": "CR-04",
    "requisito": "RF-03",
    "fecha": "2026-10-22",
    "descripcion": "Coherencia de naturaleza de cuentas PUC y consulta de saldos para todos los roles (RN-PUC-01)",
    "version_anterior": "1.0",
    "version_nueva": "1.1",
    "impacto": "Medio",
    "prioridad": "Media",
    "estado": "Aprobado"
  },
  {
    "id": "CR-05",
    "requisito": "RNF-03",
    "fecha": "2026-10-22",
    "descripcion": "Habilitar la vista de Auditoría también al rol Coordinador",
    "version_anterior": "1.0",
    "version_nueva": "1.1",
    "impacto": "Bajo",
    "prioridad": "Media",
    "estado": "Aprobado"
  },
  {
    "id": "CR-06",
    "requisito": "RF-05",
    "fecha": "2026-10-22",
    "descripcion": "Filtro por tipo de ingreso y búsqueda de texto en Todas las Causaciones",
    "version_anterior": "1.1",
    "version_nueva": "1.2",
    "impacto": "Bajo",
    "prioridad": "Baja",
    "estado": "Aprobado"
  },
  {
    "id": "CR-07",
    "requisito": "RNF-10",
    "fecha": "2026-10-22",
    "descripcion": "Nuevo requisito: guías y ayudas contextuales por pantalla y por rol",
    "version_anterior": "—",
    "version_nueva": "1.0",
    "impacto": "Medio",
    "prioridad": "Media",
    "estado": "Aprobado"
  },
  {
    "id": "CR-08",
    "requisito": "RNF-01",
    "fecha": "2026-10-23",
    "descripcion": "Reclasificar el estado real: cifrado en reposo/tránsito no implementado en modo local",
    "version_anterior": "1.0",
    "version_nueva": "1.1",
    "impacto": "Bajo",
    "prioridad": "Media",
    "estado": "Aprobado"
  },
  {
    "id": "CR-09",
    "requisito": "RNF-02",
    "fecha": "2026-10-23",
    "descripcion": "Reclasificar MFA/SSO como simulado (sin proveedor conectado)",
    "version_anterior": "1.0",
    "version_nueva": "1.1",
    "impacto": "Bajo",
    "prioridad": "Media",
    "estado": "Aprobado"
  },
  {
    "id": "CR-SEG-01",
    "requisito": "RNF-01",
    "fecha": "2026-11-12",
    "descripcion": "Contraseñas con SHA-256 sin sal en localStorage: usar hash con sal (PBKDF2) y no persistir credenciales en el navegador",
    "version_anterior": "1.1",
    "version_nueva": "1.2",
    "impacto": "Alto",
    "prioridad": "Alta",
    "estado": "Pendiente"
  }
]

df_cambios = pd.DataFrame(cambios)
display(df_cambios)

### (Opcional) Agregar las decisiones de la Parte 9
Cuando hagan la prueba con usuario, agreguen aquí cada observación como un CR:
- **aceptada** → `"Aprobado"`
- **rechazada** → `"Rechazado"`
- **pospuesta** → `"Pendiente"`

Luego ejecuten de nuevo las celdas de abajo y las métricas y gráficos se actualizan solos.

In [ ]:
cambios_usuario = [
    # {"id": "CR-10", "requisito": "RF-05", "fecha": "2026-11-__",
    #  "descripcion": "Observación del usuario ...", "version_anterior": "1.2", "version_nueva": "1.3",
    #  "impacto": "Bajo", "prioridad": "Media", "estado": "Aprobado"},
]
if cambios_usuario:
    df_cambios = pd.concat([pd.DataFrame(cambios), pd.DataFrame(cambios_usuario)], ignore_index=True)
print(f"Total de solicitudes registradas: {len(df_cambios)}")

### Métricas generadas automáticamente

In [ ]:
total = len(df_cambios)
por_estado = df_cambios["estado"].value_counts().reindex(["Aprobado", "Rechazado", "Pendiente"], fill_value=0)
por_prioridad = df_cambios["prioridad"].value_counts().reindex(["Alta", "Media", "Baja"], fill_value=0)
mas_modificados = df_cambios["requisito"].value_counts()

print(f"Cantidad de cambios : {total}")
print(f"Cambios aprobados   : {por_estado['Aprobado']}")
print(f"Cambios rechazados  : {por_estado['Rechazado']}")
print(f"Cambios pendientes  : {por_estado['Pendiente']}")

print("\nCambios por prioridad:")
display(por_prioridad.rename_axis("Prioridad").to_frame("Cambios"))

print("Requerimientos más modificados:")
top = mas_modificados[mas_modificados == mas_modificados.max()]
print("  →", ", ".join(f"{r} ({n} cambios)" for r, n in top.items()))
display(mas_modificados.rename_axis("Requisito").to_frame("Cambios"))

print("Cambios por impacto y estado:")
display(pd.crosstab(df_cambios["impacto"], df_cambios["estado"]).reindex(["Alto", "Medio", "Bajo"], fill_value=0))

### Gráficos

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
color = "#2a6f97"

# 1. Cambios por estado
ax = ejes[0]
barras = ax.bar(por_estado.index, por_estado.values, color=color)
ax.bar_label(barras, padding=3)
ax.set_title("Cambios por estado")
ax.set_ylabel("Número de cambios")

# 2. Cambios por prioridad
ax = ejes[1]
barras = ax.bar(por_prioridad.index, por_prioridad.values, color=color)
ax.bar_label(barras, padding=3)
ax.set_title("Cambios por prioridad")

# 3. Requerimientos más modificados
ax = ejes[2]
orden = mas_modificados.sort_values()
barras = ax.barh(orden.index, orden.values, color=color)
ax.bar_label(barras, padding=3)
ax.set_title("Requerimientos más modificados")
ax.set_xlabel("Número de cambios")

from matplotlib.ticker import MaxNLocator
for ax in ejes:
    ax.spines[["top", "right"]].set_visible(False)
ejes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
ejes[1].yaxis.set_major_locator(MaxNLocator(integer=True))
ejes[2].xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

### Lectura de resultados
- Casi todos los cambios se aprobaron e implementaron en un mismo ciclo (22–23 de octubre), después de la revisión del prototipo.
- **RF-02 (terceros)**, **RF-05 (causación manual)** y **RNF-01 (seguridad)** son los requerimientos más inestables: concentran la mayoría de los cambios. Son los que más necesitan criterios de aceptación claros y pruebas de regresión.
- El único cambio pendiente es **CR-SEG-01**, de impacto y prioridad altos. Debe pasar por análisis de impacto y aprobación antes de tocar el código (ver Partes 16 y 17).